In [45]:
import pandas as pd
import numpy as np

# --- STEP 1: THE BASE ROSTER (FIXED) ---

# 1. Define your target semester (presentation)
TARGET_SEMESTER = '2013J'  

# 2. Load the student demographics and final results
student_info = pd.read_csv('archive (2)/studentInfo.csv')

# 3. Apply the constraint: Filter for ALL courses, but ONLY this one semester
mask = (student_info['code_presentation'] == TARGET_SEMESTER)
base_roster = student_info[mask].copy()

# 4. Create the binary target variable (1 = Withdrawn/Fail, 0 = Pass/Distinction)
bad_outcomes = ['Withdrawn', 'Fail']
base_roster['target'] = base_roster['final_result'].isin(bad_outcomes).astype(int)

# 5. Convert IMD Band from text to a clean 1-10 numerical scale
imd_mapping = {
    '0-10%': 1, '10-20%': 2, '20-30%': 3, '30-40%': 4,
    '40-50%': 5, '50-60%': 6, '60-70%': 7, '70-80%': 8,
    '80-90%': 9, '90-100%': 10
}
base_roster['imd_numeric'] = base_roster['imd_band'].map(imd_mapping)

# 6. Create the 'course_id' to differentiate between modules in this semester
base_roster['course_id'] = base_roster['code_module'] + "_" + base_roster['code_presentation']

# 7. Keep only the core columns we need moving forward
base_roster = base_roster[['id_student', 'course_id', 'imd_numeric', 'target']]

print(f"Total students across ALL courses in {TARGET_SEMESTER}: {len(base_roster)}")
print("Missing IMD values:", base_roster['imd_numeric'].isna().sum())
print(base_roster.head())

Total students across ALL courses in 2013J: 8845
Missing IMD values: 1239
   id_student  course_id  imd_numeric  target
0       11391  AAA_2013J         10.0       0
1       28400  AAA_2013J          3.0       0
2       30268  AAA_2013J          4.0       1
3       31604  AAA_2013J          6.0       0
4       32885  AAA_2013J          6.0       0


In [46]:
# --- FIX: SURVIVORSHIP BIAS ---

# 1. Load registration data to get withdrawal dates
student_reg = pd.read_csv('archive (2)/studentRegistration.csv')
student_reg['course_id'] = student_reg['code_module'] + "_" + student_reg['code_presentation']

# 2. Merge withdrawal dates into our Base Roster
base_roster = pd.merge(
    base_roster, 
    student_reg[['id_student', 'course_id', 'date_unregistration']], 
    on=['id_student', 'course_id'], 
    how='left'
)

# 3. Filter the roster: Keep students who NEVER withdrew (NaN) OR withdrew AFTER Week 6 (>= 42)
active_w6_mask = base_roster['date_unregistration'].isna() | (base_roster['date_unregistration'] >= 42)
base_roster = base_roster[active_w6_mask].copy()

# Drop the date_unregistration column so the model doesn't use it to cheat later
base_roster = base_roster.drop(columns=['date_unregistration'])

print(f"Total students actually active at Week 6: {len(base_roster)}")

Total students actually active at Week 6: 7568


In [47]:
# --- STEP 2: THE VLE CLICKS (FIXED MERGE) ---

# 1. Load the raw click logs and the site mapping
student_vle = pd.read_csv('archive (2)/full_student_vle.csv')
vle_map = pd.read_csv('archive (2)/vle.csv')

# 2. Merge them together on ALL shared columns to prevent _x/_y suffixes
master_vle = pd.merge(
    student_vle, 
    vle_map, 
    on=['id_site', 'code_module', 'code_presentation'], 
    how='inner'
)

# 3. Create 'course_id' to match our base roster
master_vle['course_id'] = master_vle['code_module'] + "_" + master_vle['code_presentation']

# 4. Apply Semester Constraint: Keep only our target semester
master_vle = master_vle[master_vle['code_presentation'] == TARGET_SEMESTER].copy()

# 5. Apply Time Constraints: Start from Day 0, end precisely at the end of Week 6 (Day 41)
vle_w6 = master_vle[(master_vle['date'] >= 0) & (master_vle['date'] < 42)].copy()

# 6. Create accurate 14-day Time Buckets for the Recency Ratio
vle_w6['is_w1_2'] = vle_w6['date'].between(0, 13)
vle_w6['is_w5_6'] = vle_w6['date'].between(28, 41)

# 7. Aggregate clicks per student, per course
vle_agg = vle_w6.groupby(['course_id', 'id_student']).agg(
    total_clicks=('date', 'count'),              # Frequency
    clicks_w1_2=('is_w1_2', 'sum'),              # Honeymoon phase
    clicks_w5_6=('is_w5_6', 'sum'),              # Week 6 reality check
    activity_diversity=('activity_type', 'nunique') # Unique resource types used
).reset_index()

# 8. Calculate Recency Ratio (W5-6 vs W1-2)
vle_agg['recency_ratio'] = vle_agg['clicks_w5_6'] / (vle_agg['clicks_w1_2'] + 1)

# 9. Standardize: Calculate Course-Relative Percentiles
cols_to_pct = ['total_clicks', 'activity_diversity']
for col in cols_to_pct:
    pct_col_name = f'{col}_pct'
    vle_agg[pct_col_name] = vle_agg.groupby('course_id')[col].rank(pct=True)

# 10. Keep only the final standardized features
vle_features = vle_agg[['id_student', 'course_id', 'total_clicks_pct', 'recency_ratio', 'activity_diversity_pct']]

print("VLE Processing Complete.")
print(f"Total unique student-course engagements found: {len(vle_features)}")

VLE Processing Complete.
Total unique student-course engagements found: 7649


In [48]:
# --- STEP 3: THE ASSESSMENTS (LEAN VERSION) ---

student_assess = pd.read_csv('archive (2)/studentAssessment.csv')
assess_info = pd.read_csv('archive (2)/assessments.csv')
assess_info['course_id'] = assess_info['code_module'] + "_" + assess_info['code_presentation']
# This ensures we don't accidentally count scores from 2014J or 2013B
assess_info_semester = assess_info[assess_info['code_presentation'] == TARGET_SEMESTER].copy()

# 1. Look at all assignments due before Day 42
w6_assessments = assess_info_semester[assess_info_semester['date'] < 42].copy()

# ==========================================
# YOUR REQUESTED FEATURE: Does this course have assignments?
# ==========================================
course_hw = w6_assessments.groupby('course_id').agg(
    total_w6_weight=('weight', 'sum')
).reset_index()

# 1 = Yes, assignments exist for this course in the first 6 weeks
course_hw['course_has_assignments'] = 1  
# ==========================================

# 2. Match student submissions to get the weights
master_assess = pd.merge(student_assess, w6_assessments, on='id_assessment', how='inner')

# 3. Calculate Earned Weighted Score
master_assess['earned_weighted_score'] = master_assess['score'] * master_assess['weight']

# 4. Sum up the student's total earned score per course
assess_agg = master_assess.groupby(['course_id', 'id_student']).agg(
    student_total=('earned_weighted_score', 'sum')
).reset_index()

# 5. Merge the course totals into the student's record
assess_agg = pd.merge(assess_agg, course_hw, on='course_id', how='left')

# 6. Calculate True Weighted Average
import numpy as np
assess_agg['true_mean_score_w6'] = np.where(
    assess_agg['total_w6_weight'] > 0,
    assess_agg['student_total'] / assess_agg['total_w6_weight'],
    0 
)

# 7. Calculate the score percentile
assess_agg['score_pct'] = assess_agg.groupby('course_id')['true_mean_score_w6'].rank(pct=True)

# 8. Keep only the requested columns
assess_features = assess_agg[['id_student', 'course_id', 'course_has_assignments', 'score_pct']]

print("Step 3 Complete. Requested features generated.")

Step 3 Complete. Requested features generated.


In [49]:
# --- STEP 4: THE FINAL GRAND MERGE (CORRECTED) ---

# 1. Start with the Base Roster (Active Week 6 students only)
master_df = pd.merge(base_roster, vle_features, on=['id_student', 'course_id'], how='left')

# ==========================================
# 2. ATTACH THE COURSE-LEVEL HOMEWORK FLAG
# ==========================================
# We merge course_hw directly to the master dataframe based ONLY on course_id
master_df = pd.merge(master_df, course_hw[['course_id', 'course_has_assignments']], on='course_id', how='left')

# If the course wasn't in the homework table, it gets a 0 (No assignments in W6)
master_df['course_has_assignments'] = master_df['course_has_assignments'].fillna(0)

# ==========================================
# 3. ATTACH THE STUDENT-LEVEL SCORES
# ==========================================
# Now we merge the actual student percentiles
master_df = pd.merge(master_df, assess_features[['id_student', 'course_id', 'score_pct']], on=['id_student', 'course_id'], how='left')

# 4. Handle the Missing IMD Values (Create the "Unknown" tier)
master_df['imd_numeric'] = master_df['imd_numeric'].fillna(0)

# 5. Handle the "Ghosts" (Students with 0 clicks or 0 submissions)
master_df = master_df.fillna({
    'total_clicks_pct': 0,
    'recency_ratio': 0,
    'activity_diversity_pct': 0,
    'score_pct': 0
})

# 6. Prep for XGBoost: Convert course_id to a categorical type
master_df['course_id'] = master_df['course_id'].astype('category')

print("The Final Master File is Complete and mathematically sound!")
print(master_df.head())

The Final Master File is Complete and mathematically sound!
   id_student  course_id  imd_numeric  target  total_clicks_pct  \
0       11391  AAA_2013J         10.0       0          0.313333   
1       28400  AAA_2013J          3.0       0          0.566667   
2       31604  AAA_2013J          6.0       0          0.781333   
3       32885  AAA_2013J          6.0       0          0.445333   
4       38053  AAA_2013J          9.0       0          0.801333   

   recency_ratio  activity_diversity_pct  course_has_assignments  score_pct  
0       0.571429                0.508000                     1.0   0.735376  
1       0.225352                0.190667                     1.0   0.455432  
2       1.274510                0.508000                     1.0   0.515320  
3       0.122449                0.190667                     1.0   0.430362  
4       0.500000                0.190667                     1.0   0.752089  


In [43]:
base_master=master_df.copy()

In [50]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV

# Merge and handle ghosts


# 1. Define the strictly leakage-free feature set (Comma fixed!)
features = [
    'course_id', 
    'imd_numeric', 
    'total_clicks_pct', 
    'activity_diversity_pct', 
    'course_has_assignments', 
    'score_pct',                           
    'recency_ratio',
]

X = base_master[features].copy()
y = base_master['target']

# Ensure course_id is explicitly typed as categorical for XGBoost
X['course_id'] = X['course_id'].astype('category')

# 2. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Calculate Imbalance Ratio for the Base Model
num_passes = (y_train == 0).sum()
num_fails = (y_train == 1).sum()
imbalance_ratio = num_passes / num_fails

# 3. Initialize and Train Base XGBoost
print("Training Base XGBoost Model...")
base_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.04,
    max_depth=5,
    scale_pos_weight=imbalance_ratio, # This handles the class imbalance internally
    tree_method="hist",               
    enable_categorical=True,          
    eval_metric='auc',
    random_state=42
)

# 4. Calibrate the Probabilities
print("Applying Isotonic Calibration...")
calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv=5)
calibrated_model.fit(X_train, y_train)

# Generate Probabilities on the UNSEEN Test Set
y_probs = calibrated_model.predict_proba(X_test)[:, 1]

# ==========================================
# 5. THE SIMPLE ALTERNATIVE: MANUAL THRESHOLD
# ==========================================
# Change this single number to tune your model manually.
# Lowering it catches more failing students (Higher Recall).
# Raising it reduces false alarms (Higher Precision).
MANUAL_THRESHOLD = 0.35

print(f"Applying Manual Cutoff Threshold at {MANUAL_THRESHOLD}...")
y_pred_custom = (y_probs >= MANUAL_THRESHOLD).astype(int)

# ==========================================
# 6. FINAL REPORT
# ==========================================
print("\n" + "-" * 50)
print(f"CLASSIFICATION REPORT (Manual Threshold = {MANUAL_THRESHOLD})")
print("-" * 50)
print(classification_report(y_test, y_pred_custom, target_names=['Pass (0)', 'At Risk (1)']))

print("-" * 50)
print("CONFUSION MATRIX")
print("-" * 50)
print(confusion_matrix(y_test, y_pred_custom))

roc_auc = roc_auc_score(y_test, y_probs)
print("-" * 50)
print(f"🚀 ROC-AUC SCORE: {roc_auc:.4f}")
print("-" * 50)

Training Base XGBoost Model...
Applying Isotonic Calibration...
Applying Manual Cutoff Threshold at 0.35...

--------------------------------------------------
CLASSIFICATION REPORT (Manual Threshold = 0.35)
--------------------------------------------------
              precision    recall  f1-score   support

    Pass (0)       0.77      0.67      0.72       895
 At Risk (1)       0.60      0.72      0.65       619

    accuracy                           0.69      1514
   macro avg       0.68      0.69      0.68      1514
weighted avg       0.70      0.69      0.69      1514

--------------------------------------------------
CONFUSION MATRIX
--------------------------------------------------
[[596 299]
 [176 443]]
--------------------------------------------------
🚀 ROC-AUC SCORE: 0.7627
--------------------------------------------------


In [51]:
import numpy as np
from sklearn.calibration import calibration_curve

# --- 1. TOP 3 FEATURE EXTRACTION (PATCHED) ---
print("--- TOP 3 RISK DRIVERS ---")
# Fit the base model directly to expose the internal XGBoost trees
base_model.fit(X_train, y_train)

importances = base_model.feature_importances_
indices = np.argsort(importances)[::-1]

for i in range(3):
    feature_name = features[indices[i]]
    importance_score = importances[indices[i]]
    print(f"{i+1}. {feature_name} (Impact Score: {importance_score:.4f})")

# --- 2. CALIBRATION ANALYSIS ---
print("\n--- CALIBRATION ANALYSIS ---")
print("Does the predicted risk match reality?")
# We split the predictions into 5 buckets
prob_true, prob_pred = calibration_curve(y_test, y_probs, n_bins=5, strategy='uniform')

for pred, true in zip(prob_pred, prob_true):
    print(f"When the model predicts ~{pred*100:.0f}% risk  -->  Actual failure rate is {true*100:.0f}%")

--- TOP 3 RISK DRIVERS ---
1. course_id (Impact Score: 0.2298)
2. total_clicks_pct (Impact Score: 0.1874)
3. score_pct (Impact Score: 0.1703)

--- CALIBRATION ANALYSIS ---
Does the predicted risk match reality?
When the model predicts ~16% risk  -->  Actual failure rate is 15%
When the model predicts ~28% risk  -->  Actual failure rate is 27%
When the model predicts ~49% risk  -->  Actual failure rate is 51%
When the model predicts ~68% risk  -->  Actual failure rate is 64%
When the model predicts ~91% risk  -->  Actual failure rate is 92%
